In [2]:
import emcee
import numpy as np

# 1. Configure your file path and burn-in
# BACKEND_PATH = "../data/posteriors/forecast_w0waCDM_fixed_scatter/lsst_y1.h5"
BACKEND_PATH = "../data/posteriors/forecast_w0waCDM_free_scatter/lsst_y10.h5"
N_BURN = 1000  # Number of initial steps to discard as burn-in

# 2. Load the backend reader
reader = emcee.backends.HDFBackend(BACKEND_PATH, read_only=True)

# 3. Fetch basic chain properties
n_steps_total = reader.iteration
n_walkers, ndim = reader.shape  # <-- FIXED: Extract from the shape tuple
n_steps_kept = n_steps_total - N_BURN

print(f"File: {BACKEND_PATH}")
print(f"Total steps recorded: {n_steps_total}")
print(f"Number of walkers:   {n_walkers}")
print(f"Steps kept (post-burn): {n_steps_kept}\n")

print("--- AUTOCORRELATION TIME (tau) ---")
print(f"{'Param Index':<12} | {'tau (steps)':<12} | {'Min Required':<15} | {'Status'}")
print("-" * 55)

try:
    # emcee can compute tau directly from the backend reader
    tau = reader.get_autocorr_time(discard=N_BURN, tol=0)
    max_tau = np.max(tau)

    for i, t_val in enumerate(tau):
        min_steps = int(50 * t_val)
        status = "✓ OK" if n_steps_kept > min_steps else "! SHORT"
        print(f"Param {i:<7} | {t_val:<12.1f} | {min_steps:<15} | {status}")

    print("-" * 55)
    if n_steps_kept > 50 * max_tau:
        print(f"OVERALL STATUS: ✓ CONVERGED")
    else:
        print(f"OVERALL STATUS: ! WARNING: Unconverged (Needs {int(50*max_tau)} post-burn steps, only has {n_steps_kept})")

except Exception as e:
    print(f"OVERALL STATUS: ERROR calculating tau: {e}")
    print("This usually happens if the chain is far too short or completely stuck.")

File: ../data/posteriors/forecast_w0waCDM_free_scatter/lsst_y10.h5
Total steps recorded: 15000
Number of walkers:   64
Steps kept (post-burn): 14000

--- AUTOCORRELATION TIME (tau) ---
Param Index  | tau (steps)  | Min Required    | Status
-------------------------------------------------------
Param 0       | 347.3        | 17362           | ! SHORT
Param 1       | 253.1        | 12656           | ✓ OK
Param 2       | 300.1        | 15005           | ! SHORT
Param 3       | 239.6        | 11980           | ✓ OK
Param 4       | 339.8        | 16988           | ! SHORT
Param 5       | 252.3        | 12616           | ✓ OK
Param 6       | 365.8        | 18290           | ! SHORT
Param 7       | 217.8        | 10889           | ✓ OK
Param 8       | 208.9        | 10445           | ✓ OK
-------------------------------------------------------
OVERALL STATUS: ! WARNING: Unconverged (Needs 18290 post-burn steps, only has 14000)


In [11]:
# Assuming 'reader' is your HDFBackend object from the previous cell
acceptance_fraction = np.mean(reader.accepted / reader.iteration)
print(f"Mean acceptance fraction: {acceptance_fraction:.3f}")

Mean acceptance fraction: 0.322
